# Import

In [1]:
import pandas as pd
from scipy.io import loadmat
import pandas as pd
import re
import networkx as nx
import numpy as np
from collections import OrderedDict

In [39]:
import torch
import transformers
import numpy as np
import wandb
from datasets import load_dataset, Dataset
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead, create_reference_model
from typing import List
from tqdm import tqdm
import copy
from transformers import AutoModelForCausalLM, AutoTokenizer

from utils.prompting import *
from utils.utils import write_json, append_jsonl, normalize_answer, set_seed, load_timeqax_data
import fire
wandb.init(mode="disabled")
import pandas as pd

from peft import (
    PeftModel,
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

import re
import statistics
import random
import time
import gc
import torch

# Read Data

In [3]:
merged_df = pd.read_pickle("../data/test/test.pkl")

In [4]:
print(merged_df['s5_annotated_personalized_summaries'].iloc[1])

Alright, fellow grooming adventurers, let’s talk about the Gillette Mach 3 razor — the trusty sidekick that promises to turn your shaving saga from “ouch” to “ahh” with a few slick moves.

First off, swapping blades on this bad boy is a breeze thanks to its single-point cartridge docking system. No fumbling, no finger gymnastics—just a quick click and you’re back in the game. And if you’re like me, tired of razors that clog faster than your inbox, the Mach 3’s design keeps hair from gumming up the works, making cleanup a snap. Bonus points for saving you from that awkward “why is this razor so gross?” moment.

Now, the real magic happens with its swivel head and spring mechanism. It’s like the razor has a built-in GPS for your face, hugging every contour with the grace of a ballerina—well, a ballerina who’s really into smooth skin. This means fewer nicks and less irritation, even if your skin screams “sensitive!” at the mere sight of a blade. The three-blade system glides so effortless

In [5]:
merged_df.iloc[1]

category                                                                          Beauty
product_name                                                       Gillette Mach 3 Razor
product_reviews                        [    ~ ~ OK girls. This opinion is really for ...
user_id                                                                          5000858
filtered_hist_vote_written             [    I bought these for going to festivals in ...
s1_kp                                  [The Gillette Mach 3 razor provides an excepti...
s2_kp_helpfulness_cal                  [{'kp': 'Changing blades on the Mach 3 is quic...
s3_user_profiles                       ### User Profile Summary\n\n**Personality Trai...
s4_helpful_kps_filtered                [Changing blades on the Mach 3 is quick and si...
s5_annotated_personalized_summaries    Alright, fellow grooming adventurers, let’s ta...
Name: 1, dtype: object

In [46]:
test_df = merged_df

# Read Model Checkpoint

In [6]:
TEMPLATE = get_prompt("helpfulsumm_cot_helpful_pos")

In [7]:
device_map = "auto"
world_size = int(os.environ.get("WORLD_SIZE", 1))
ddp = world_size != 1
if ddp:
    device_map = {"": int(os.environ.get("LOCAL_RANK") or 0)}

In [8]:
base_model = "hugging-quants/Meta-Llama-3.1-8B-Instruct-GPTQ-INT4"
source_path = '../models/stage_2_helpfulsumm_rl/step-200/'

In [9]:
def set_seed(seed, n_gpu=1):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [16]:
# training hyperparams
batch_size = 1
num_epochs = 1
learning_rate = 2e-7
lora_r = 64
lora_alpha = 16
lora_dropout = 0.05
lora_target_modules = ['q_proj', 'v_proj', 'k_proj', 'o_proj']
train_on_inputs = False  # if False, masks out inputs in loss
add_eos_token = False
eval_steps=200
save_steps=200
save_total_limit=10
seed=201
debug_mode = True

In [17]:
set_seed(seed=seed)

In [18]:
policy_model = AutoModelForCausalLM.from_pretrained(
    source_path,
    device_map='auto',
    trust_remote_code=False,
    revision="main",
    # attn_implementation="flash_attention_2"
)

/mnt/c/Users/antan/Desktop/PHD READING/Product_Question_Answering/JustiLM_Question_based_Summarization/AutoGPTQ/auto_gptq/nn_modules/triton_utils/kernels.py:411: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, qweight, scales, qzeros, g_idx, bits, maxq):
/mnt/c/Users/antan/Desktop/PHD READING/Product_Question_Answering/JustiLM_Question_based_Summarization/AutoGPTQ/auto_gptq/nn_modules/triton_utils/kernels.py:419: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/mnt/c/Users/antan/Desktop/PHD READING/Product_Question_Answering/JustiLM_Question_based_Summarization/AutoGPTQ/auto_gptq/nn_modules/triton_utils/kernels.py:461: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` i

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/peft/utils/save_and_load.py:328: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  adapters_weights = to

In [19]:
from auto_gptq import exllama_set_max_input_length
policy_model = exllama_set_max_input_length(policy_model, max_input_length=65536)

In [20]:
policy_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (rotary_emb): LlamaRotaryEmbedding()
          (k_proj): lora.QuantLinear(
            (base_layer): QuantLinear()
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=4096, out_features=64, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=64, out_features=1024, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (quant_linear_module): QuantLinear()
          )
          (o_proj): lora.QuantLinear(
            (base_layer): QuantLinear()
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inpla

In [21]:
peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    target_modules=lora_target_modules,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)

In [22]:
ref_model = create_reference_model(policy_model)
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(ref_model)

In [23]:
policy_model = get_peft_model(policy_model, peft_config)
policy_model = AutoModelForCausalLMWithValueHead.from_pretrained(policy_model)

In [24]:
tokenizer = transformers.AutoTokenizer.from_pretrained(
        base_model,
        padding_side="right",
    )
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = (
    tokenizer.eos_token_id
)
tokenizer.add_prefix_space = False

In [25]:
def tokenize(prompt):
    result = tokenizer(
        prompt,
    )
    result["labels"] = result["input_ids"].copy()

    return result

In [26]:
def generate_and_tokenize_prompt(data_point, hist_vote_written_size=5):
    product_name = data_point['product_name']
    product_reviews = data_point['product_reviews']
    if len(product_reviews) > 30:
        import random
        random.seed(42)
        product_reviews = random.sample(product_reviews, 30)

    import random
    random.seed(42)
    hist_vote_written = random.sample(data_point['filtered_hist_vote_written'], min(hist_vote_written_size, len(data_point['filtered_hist_vote_written'])))
    
    query = TEMPLATE.format(product_name = product_name, product_reviews = product_reviews, 
                            hist_vote_written = hist_vote_written)
    
    msg = [{"role": "user", "content": query}]
    formatted_prompt =  tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    tokenized_full_prompt = tokenize(formatted_prompt)
    
    return tokenized_full_prompt

def generate_and_tokenize_prompt_gen(data_point, hist_vote_written_size):
    return generate_and_tokenize_prompt(data_point, hist_vote_written_size)

def process_query_tensor(qt):
    i = qt.tolist().index(128000)
    return qt[i:]

In [27]:
if not ddp and torch.cuda.device_count() > 1:
    # keeps Trainer from trying its own DataParallelism when more than 1 gpu is available
    ref_model.is_parallelizable = True
    ref_model.model_parallel = True
    policy_model.is_parallelizable = True
    policy_model.model_parallel = True

In [28]:
micro_batch_size = 1
_batch_size = micro_batch_size
gradient_accumulation_steps = 1
_batch_size = micro_batch_size
_lr = learning_rate
config = PPOConfig(
    reward_model=None,
    kl_penalty="kl",
    batch_size=2,
    mini_batch_size=1,
    gradient_accumulation_steps=gradient_accumulation_steps,
    ppo_epochs=num_epochs,
    learning_rate=_lr,
    remove_unused_columns=False,
    seed=42,
)

trainer = PPOTrainer(
    config=config,
    tokenizer=tokenizer,
    model=policy_model,
    ref_model=ref_model,
#     dataset=train_data,
    data_collator=transformers.DataCollatorForSeq2Seq(
            tokenizer,
            pad_to_multiple_of=8,
            return_tensors="pt",
            padding=True,
        ),
)

generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": False,
    "repetition_penalty": 1.0,
    "pad_token_id": tokenizer.eos_token_id,
    "max_new_tokens": 1000,
#     "max_new_tokens": 15,
    "eos_token_id": tokenizer.eos_token_id,
}

/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/trl/trainer/ppo_trainer.py:257: UserWarning: No dataset is provided. Make sure to set config.batch_size to the correct value before training.
  warnings.warn(


# Inference

## Setup

In [29]:
from transformers import AutoTokenizer, AutoModelForCausalLM
BASE = "hugging-quants/Meta-Llama-3.1-8B-Instruct-GPTQ-INT4"
tokenizer = AutoTokenizer.from_pretrained(BASE)

In [31]:
for col in ['product_reviews', 'filtered_hist_vote_written']:
    merged_df[col] = merged_df[col].apply(lambda x: [rev.strip() for rev in x])

## Trial

In [33]:
row = merged_df.iloc[1]

In [34]:
def summary_generation(row, hist_vote_written_size):
    test_data = pd.DataFrame([row.to_dict()])
    test_data = Dataset.from_pandas(test_data).map(generate_and_tokenize_prompt_gen, fn_kwargs={"hist_vote_written_size": hist_vote_written_size})
    test_data = test_data.select_columns(['input_ids', 'attention_mask', 'labels'])
    test_data.set_format("torch")
    from torch.utils.data import DataLoader
    test_data = DataLoader(test_data, batch_size=2)
    
    for batch in test_data:
        query_tensors = batch["input_ids"]
        input_tensors_b = [process_query_tensor(qt) for qt in query_tensors]
        response_tensors_b = []
        for input_tensors in input_tensors_b:
            response_tensors = trainer.generate([input_tensors], return_prompt=False, **generation_kwargs)
            response_tensors_b += response_tensors
        
        response_b = [tokenizer.decode(rt, skip_special_tokens=True).strip() for rt in response_tensors_b]
        
    del query_tensors, input_tensors_b, input_tensors, response_tensors, response_tensors_b
    gc.collect()
    torch.cuda.empty_cache()  # harmless; can help reduce fragmentation
        
    return response_b[0]

In [41]:
def personanlized_summarize(row):
    generated_responses = []
    for hist_sample_size in range(5, 35, 5):
        try:
            response = summary_generation(row, hist_sample_size)
            generated_responses += [response]

            ext = re.findall(r"Personalized [Ss]ummary(?:\]|[^:\n]*:)[ \n]*((?:.+\n*)+)", response)  # MAY BE CAN TRY THIS
            if len(ext) > 0:
                summary = ext[0]
                return summary, generated_responses
            time.sleep(3)
        except Exception as e:
            print(e)
        
    return None, generated_responses

In [52]:
row = test_df.iloc[3]
print(len(row['filtered_hist_vote_written']))
row

20


category                                                                          Beauty
product_name                                                       Gillette Mach 3 Razor
product_reviews                        [~ ~ OK girls. This opinion is really for the ...
user_id                                                                          5050855
filtered_hist_vote_written             [i love alberto balsams esspecially the sunkis...
s1_kp                                  [The Gillette Mach 3 razor provides an excepti...
s2_kp_helpfulness_cal                  [{'kp': 'Replacement blades for the Mach 3 are...
s3_user_profiles                       ### User Profile Summary\n\n**Personality Trai...
s4_helpful_kps_filtered                [Replacement blades for the Mach 3 are relativ...
s5_annotated_personalized_summaries    If you’re weighing the Gillette Mach 3 razor, ...
Name: 3, dtype: object

In [43]:
%%time
summary, responses = personanlized_summarize(row)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


CPU times: user 31.9 s, sys: 6.15 s, total: 38.1 s
Wall time: 39.7 s


In [44]:
print(summary)

Based on the user profile and the helpful key points, here is a personalized summary of the Gillette Mach 3 Razor tailored to user 111:

As a seasoned shaver, you know that a good razor can make all the difference in your daily routine. The Gillette Mach 3 Razor is a top-of-the-line product that delivers a close and comfortable shave every time. With its three-blade system, lubricating strip, and ergonomic design, this razor is designed to provide a smooth and close shave, even for users with sensitive skin.

While the replacement blades may be a bit pricey, they last a long time, making it a worthwhile investment for those who value a good shave. The razor is also easy to use and clean, making it a great option for those who want a hassle-free shaving experience.

Overall, the Gillette Mach 3 Razor is a top-notch product that delivers on its promises. If you're looking for a razor that will provide a close and comfortable shave, look no further than the Mach 3.

As a user who values a

## Inference

In [56]:
test_df = test_df[['category', 'product_name', 'user_id', 'product_reviews', 'filtered_hist_vote_written']]

In [58]:
test_df['my_category'] = 1

/tmp/ipykernel_12569/712584442.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['my_category'] = 1


In [59]:
root_path = f"../output/stage_2_rl_inference/summary"

inputs = [(root_path,
           domain,
           test_df[test_df['my_category'] == domain].reset_index(drop=True)
           )
          for domain in test_df['my_category'].unique()]

In [60]:
num_workers = 1

In [61]:
from datasets import concatenate_datasets, load_dataset
from datasets import Dataset, DatasetDict
import pandas as pd
import numpy as np
import torch
import os
import ast
import time
from multiprocessing import Pool
from pathlib import Path
from os import listdir
from tqdm import tqdm
import random
import re
import math

In [62]:
domain = 1 
domain_df = test_df.reset_index(drop=True)
save_step=1

In [126]:
src_path = f"{root_path}/{domain}"
Path(src_path).mkdir(parents=True, exist_ok=True)
personalized_summaries = []

file_names = listdir(src_path)
postfix = [re.split("[_.]", name)[1]
           for name in listdir(src_path)
           ]
start = 0

for i, (_, row) in tqdm(enumerate(domain_df.iterrows()), total=domain_df.shape[0]):
    if i < start:
        continue

    summary, responses = personanlized_summarize(row)
    personalized_summaries += [(summary, responses)]
    time.sleep(3)

    if (i + 1) % save_step == 0:
        save_df = domain_df.iloc[:i + 1]
        save_df.insert(0, 'personalized_summaries', personalized_summaries)
        save_df.to_pickle(f"{src_path}/{domain}_{i + 1}.pkl")

new_domain_df = domain_df.iloc[:i + 1]
new_domain_df.insert(0, 'personalized_summaries', personalized_summaries)
new_domain_df.to_pickle(f"{src_path}/{domain}_done.pkl")

  0%|                                                                                                                                                                                     | 0/100 [00:00<?, ?it/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
  1%|█▋                                                                                                                                                                           | 1/100 [00:28<46:57, 28.46s/it]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


...

/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
 98%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌   | 98/100 [1:31:22<01:56, 58.09s/it]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
 99%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎ | 99/100 [1:31:57<00:51, 51.30s/it]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [1:32:56<00:00, 55.76s/it]


## Read

In [64]:
root_path = f"../output/stage_2_rl_inference/summary"
personalized_summary_df = pd.read_pickle(root_path + "/1/1_done.pkl")
mask = pd.isnull(personalized_summary_df['personalized_summaries'].apply(lambda x: x[0]))
personalized_summary_df[mask]

,personalized_summaries,category,product_name,voter_full,product_reviews,hist_vote_written,filtered_hist_vote_written,my_category
18,"(None, [# User Profile Analysis\n\nBased on th...",Computers,HP Deskjet 930c,4295,[ The time had come. My little cheap printe...,"[ Okay, so we had zip and then we got zip 2...","[ Okay, so we had zip and then we got zip 2...",1
55,"(None, [**User Profile Analysis**\n\nBased on ...",Games,Nintendo GameBoy Advance,5050835,[ Bought my Gameboy Advance yesterday from ...,[ Through all the hype of the film and all ...,[ Through all the hype of the film and all ...,1
82,"(None, [# User Profile Analysis\n\nBased on th...",Shopping,Co-op (Shop),5241222,[ My local Co-Op Swift shop opened a coup...,[ Well asda just gets better and batter sin...,[ Well asda just gets better and batter sin...,1
86,"(None, [# User Profile Analysis\n\nBased on th...",Software,Microsoft Windows Millennium Edition - complet...,15574,[ My husband is a computer geek and can pro...,[ There are very few of us who can say one ...,[ MSN Messenger is a messenger service whic...,1


In [65]:
personalized_summary_df = personalized_summary_df[~mask]

In [66]:
personalized_summary_df['personalized_summaries'] = personalized_summary_df['personalized_summaries'].apply(lambda x: x[0])

In [67]:
personalized_summary_df['personalized_summaries'] = personalized_summary_df['personalized_summaries'].apply(
    lambda x: re.sub(r"\n*Note:[^\n]+", "", x).strip("\n").strip())

In [68]:
personalized_summary_df['personalized_summaries'].iloc[6]

'Based on the user\'s profile and the helpful key points, the following summary is generated:\n\n"Life Expectancy" by Dean Koontz is a thrilling page-turner that will keep you on the edge of your seat. The story follows the life of Jimmy Tock, a pastry chef\'s son who is born with a strange destiny thrust upon him. As Jimmy navigates through his life, he experiences five terrible days that are predicted by his grandfather, Josef Tock. The book is full of suspense, twists, and turns, and the banter between Jimmy and Lorrie Lynn Hicks is a highlight of the story. The book is not too frightening but still holds suspense throughout, making it a great read for fans of Dean Koontz. If you\'re looking for a book that reminds you of the old Dean Koontz style, then "Life Expectancy" is a great choice.\n\nThe summary is written in a style that is consistent with the user\'s profile, with a focus on the suspenseful and thrilling aspects of the book. The summary also highlights the well-developed 

## Claim Extraction

In [69]:
from openai import OpenAI
client = OpenAI(
    api_key = "<YOUR API KEY HERE"
)
model="gpt-4.1"

In [70]:
def get_completion(prompt, model=model):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=1000,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

In [71]:
base_prompt = get_prompt("summary_kp_extraction")

### Sample

In [72]:
row = personalized_summary_df.iloc[1]

print(row['personalized_summaries'])
prompt = base_prompt %(row['personalized_summaries'])

Based on the user profile and the helpful key points, here is a personalized summary of the Gillette Mach 3 Razor that may be helpful to user 111:

The Gillette Mach 3 Razor is a high-quality razor that provides a close and comfortable shave. With its three blades and lubricating strip, it is easy to use and provides a smooth shave. The razor is durable and the blades last a long time, making it a good value for the price. The razor is also easy to clean and maintain, making it a great choice for those who want a hassle-free shaving experience.

As a user who is particular about getting a close shave, you will appreciate the effectiveness of the Mach 3 Razor. The razor is designed to provide a close shave with minimal irritation, making it a great choice for those with sensitive skin. The razor is also easy to use, with a comfortable grip and a simple design that makes it easy to maneuver.

Overall, the Gillette Mach 3 Razor is a great choice for anyone who wants a high-quality shave. 

In [141]:
response = get_completion(prompt, model)

In [142]:
print(response)

```json
[
  "The Gillette Mach 3 Razor provides a close and comfortable shave.",
  "The razor is easy to use and provides a smooth shave.",
  "The razor is durable and the blades last a long time.",
  "The razor is a good value for the price.",
  "The razor is easy to clean and maintain.",
  "The razor is designed to provide a close shave with minimal irritation.",
  "The razor is a great choice for those with sensitive skin.",
  "The razor has a comfortable grip and a simple design that makes it easy to maneuver."
]
```


### Run

In [73]:
def get_claim_split_completion(personalized_summaries):
    prompt = base_prompt %(personalized_summaries)
    
    retries = 5
    while retries > 0:
        try:
            response = get_completion(prompt, model)
            return response
        except Exception as e:
            if e:
                if "exceeded your current quota" in str(e).lower():
                    raise e
                print(e)
                print('Timeout error, retrying...')
                retries -= 1
                if "limit reached for" in str(e).lower():
                    time.sleep(30)
                else:
                    time.sleep(5)
            else:
                raise e

    print('API is not responding, moving on...')
    return None

In [74]:
def prompted_claim_split_generation(root_path, domain, domain_df, save_step=10):
    src_path = f"{root_path}/{domain}"
    Path(src_path).mkdir(parents=True, exist_ok=True)
    claim_split_predicted_list = []

    file_names = listdir(src_path)
    postfix = [re.split("[_.]", name)[1]
               for name in listdir(src_path)
               ]
    start = 0
    if 'done' in postfix:
        print(domain, ": ", "Loaded saved file. Done")
        new_domain_df = pd.read_pickle(f"{src_path}/{domain}_done.pkl")
        return new_domain_df
    elif len(postfix) > 0:
        last_index = max([int(idx) for idx in postfix if idx != 'done'])
        last_domain_df = pd.read_pickle(f"{src_path}/{domain}_{last_index}.pkl")
        claim_split_predicted_list = last_domain_df['claim_split_predicted'].tolist()
        start = last_index
        print(domain, "Loaded saved file. Continuing")
    else:
        print(domain, "Start new process.")

    for i, (_, row) in tqdm(enumerate(domain_df.iterrows()), total=domain_df.shape[0]):
        if i < start:
            continue

        personalized_summaries = row['personalized_summaries']
        claim_split_predicted = get_claim_split_completion(personalized_summaries)
        claim_split_predicted_list += [claim_split_predicted]
        time.sleep(0.1)
        
        if (i + 1) % save_step == 0:
            save_df = domain_df.iloc[:i + 1]
            save_df.insert(0, 'claim_split_predicted', claim_split_predicted_list)
            save_df.to_pickle(f"{src_path}/{domain}_{i + 1}.pkl")

    new_domain_df = domain_df.iloc[:i + 1]
    new_domain_df.insert(0, 'claim_split_predicted', claim_split_predicted_list)
    new_domain_df.to_pickle(f"{src_path}/{domain}_done.pkl")
    return new_domain_df

In [75]:
personalized_summary_df['my_category'] = 1

In [76]:
root_path = f"../output/stage_2_rl_inference/summary_kp_extraction"
inputs = [(root_path,
           domain,
           personalized_summary_df[personalized_summary_df['my_category'] == domain].reset_index(drop=True)
           )
          for domain in personalized_summary_df['my_category'].unique()]

In [77]:
num_workers = 1

In [80]:
from datasets import concatenate_datasets, load_dataset
from datasets import Dataset, DatasetDict
import pandas as pd
import numpy as np                                                                                                                                                                      
import torch
import os
import ast
import time
from multiprocessing import Pool
from pathlib import Path
from os import listdir
from tqdm import tqdm
import random
import re
import math

In [132]:
start_time = time.time()
with Pool(num_workers) as processor:
    data = processor.starmap(prompted_claim_split_generation, inputs)
print("TIME ELAPSED", time.time() - start_time)

1 Start new process.


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 96/96 [03:17<00:00,  2.06s/it]


TIME ELAPSED 200.33539652824402


### Read

In [83]:
root_path = f"../output/stage_2_rl_inference/summary_kp_extraction"
claim_summary_df = pd.read_pickle(root_path + "/1/1_done.pkl")

In [84]:
claim_summary_df['personalized_summaries'].iloc[0]

"Here is a personalized summary of product A (Head & Shoulders Normal Hair Shampoo) tailored to user 111's preferences:\n\n**Effective Dandruff Control**\n\nHead & Shoulders Normal Hair Shampoo is a reliable and effective product for controlling dandruff and itchiness. With its unique formula, it leaves the hair soft, clean, and shiny, making it a great choice for users with dry and curly hair like user 111.\n\n**Easy to Use**\n\nThis shampoo is easy to use and comes in a range of styles, including shampoo and condition in one, and shampoo especially for greasy hair. Its fresh fragrance lasts for a long time, making it a great choice for users who want a pleasant shower experience.\n\n**Practical and Affordable**\n\nHead & Shoulders Normal Hair Shampoo is priced reasonably, around £3-£4 for a 200ml bottle. It is also available in various sizes, making it a great choice for users who want a convenient and affordable option.\n\n**Important Note**\n\nHowever, it is essential to note that 

In [85]:
claim_summary_df['claim_split_predicted'] = claim_summary_df['claim_split_predicted'].apply(lambda x: x.strip("```json").strip("\n"))

In [86]:
mask = claim_summary_df['claim_split_predicted'].str.contains('Please provide the personalized summary')
claim_summary_df = claim_summary_df[~mask]

In [87]:
claim_summary_df['claim_split_predicted'] = claim_summary_df['claim_split_predicted'].apply(lambda x: ast.literal_eval(x))

In [93]:
claim_summary_df

,claim_split_predicted,personalized_summaries,category,product_name,voter_full,product_reviews,hist_vote_written,filtered_hist_vote_written,my_category
0,[Head & Shoulders Normal Hair Shampoo is effec...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1
1,[The Gillette Mach 3 Razor provides a close an...,Based on the user profile and the helpful key ...,Beauty,Gillette Mach 3 Razor,5000858,[ ~ ~ OK girls. This opinion is really for ...,[ When I had my red low lights done a coupl...,[ I bought these for going to festivals in ...,1
2,[Pitrok is a natural deodorant free from artif...,Based on the helpful key points and the user p...,Beauty,Pitrok,5296801,[ this is the natural option to fight per...,[ 8776; Dandruff - the evil flaky whit...,[ 02 is I think one of the best innovations...,1
3,[The Gillette Mach 3 Razor delivers a close an...,Here is a personalized summary of product A (G...,Beauty,Gillette Mach 3 Razor,5050855,[ ~ ~ OK girls. This opinion is really for ...,[ They are the 23.30.From five days I don t...,[ i love alberto balsams esspecially the su...,1
4,"[The shampoo is gentle and effective., It is a...",Based on the user profile and the helpful key ...,Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,[ Searching the freebie sites I found sache...,"[ Has a lovely light smell, really pretty a...","[ Has a lovely light smell, really pretty a...",1
...,...,...,...,...,...,...,...,...,...
91,[Dollar Rent A Car Worldwide is a reliable and...,Based on the helpful key points and user 111's...,Travel,Dollar Rent A Car Worldwide,5297771,[ AH THE DREAM.- The thrill of the open ro...,[ AH THE DREAM.- The thrill of the open ro...,[ AH THE DREAM.- The thrill of the open ro...,1
92,[Amsterdam is steeped in tradition and is very...,Here is a personalized summary of product A (A...,Travel,Amsterdam (Netherlands),5202501,[ A week off of University and online trav...,[ Air Canada isn t that bad of an airline. ...,[ Air Canada isn t that bad of an airline. ...,1
93,[Leicester offers a diverse range of shopping ...,Here is a personalized summary of product A (L...,Travel,Leicester in General,5020891,"[ Having lived here for 25 years now, this ...",[ Visitors to London who get the London Pas...,[ Visitors to London who get the London Pas...,1
94,[Milan features stunning architecture that imp...,Milan is a city that will leave you in awe. Fr...,Travel,Milan in general,5091015,[ Milculo is a swear word I was born near M...,[ Milan undoubtedly has some of the finest ...,[ Milan undoubtedly has some of the finest ...,1
